# CELL 1 — Install packages

In [1]:
!pip install matplotlib seaborn pandas numpy scipy -q
print("✅ Packages installed")

✅ Packages installed


# CELL 2 — Mount Drive + Imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

import pandas as pd
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import matplotlib.patches as mpatches
from matplotlib.colors import LinearSegmentedColormap
import seaborn as sns
from scipy import stats
import os, warnings
warnings.filterwarnings('ignore')

# ── Paths
DRIVE_PATH  = '/content/drive/MyDrive/PoliGraphX'
OUTPUT_PATH = os.path.join(DRIVE_PATH, 'outputs')
os.makedirs(OUTPUT_PATH, exist_ok=True)

# ── Color Palette (vibrant)
LEFT_COLOR    = '#00B0FF'   # Bright Blue
RIGHT_COLOR   = '#FF1744'   # Bright Red
POS_COLOR     = '#00E676'   # Bright Green
NEG_COLOR     = '#FF6D00'   # Bright Orange
NEUTRAL_COLOR = '#E040FB'   # Purple
BG_COLOR      = '#0A0A1A'   # Dark background
GRID_COLOR    = '#1E1E3A'   # Grid lines

plt.rcParams.update({
    'figure.facecolor'  : BG_COLOR,
    'axes.facecolor'    : BG_COLOR,
    'axes.edgecolor'    : '#333355',
    'axes.labelcolor'   : 'white',
    'xtick.color'       : 'white',
    'ytick.color'       : 'white',
    'text.color'        : 'white',
    'grid.color'        : GRID_COLOR,
    'grid.alpha'        : 0.5,
})

print("✅ Imports done")


Mounted at /content/drive
✅ Imports done


# CELL 3 — Load all datasets

In [3]:
print("Loading datasets...")

# Main labeled dataset
df = pd.read_csv(os.path.join(DRIVE_PATH, 'tweets_sentiment_labeled.csv'), low_memory=False)
df = df[df['clean_text'].notna()].copy()
df = df.reset_index(drop=True)

# Network summary
net_summary_path = os.path.join(OUTPUT_PATH, 'network_summary.csv')
net_metrics_path = os.path.join(OUTPUT_PATH, 'network_metrics.csv')

net_summary = pd.read_csv(net_summary_path) if os.path.exists(net_summary_path) else None
net_metrics = pd.read_csv(net_metrics_path) if os.path.exists(net_metrics_path) else None

# Parse dates
if 'date' in df.columns:
    df['date']  = pd.to_datetime(df['date'], errors='coerce')
    df['month'] = df['date'].dt.to_period('M')
    df['year']  = df['date'].dt.year
    has_date    = True
else:
    has_date = False

# Majority vote sentiment
sentiment_cols = ['vader_sentiment', 'textblob_sentiment', 'roberta_sentiment']
available_cols = [c for c in sentiment_cols if c in df.columns]

def majority_vote(row):
    votes = [row[c] for c in available_cols]
    pos   = votes.count('Positive')
    neg   = votes.count('Negative')
    return 'Positive' if pos >= neg else 'Negative'

df['final_sentiment'] = df.apply(majority_vote, axis=1)

print(f"Total tweets    : {len(df):,}")
print(f"Ideology dist   : {df['ideology'].value_counts().to_dict()}")
print(f"Sentiment dist  : {df['final_sentiment'].value_counts().to_dict()}")
print(f"Available sent  : {available_cols}")
print(f"Date available  : {has_date}")
print("✅ All data loaded")

Loading datasets...
Total tweets    : 42,110
Ideology dist   : {'Right': 26866, 'Left': 15244}
Sentiment dist  : {'Positive': 23662, 'Negative': 18448}
Available sent  : ['vader_sentiment', 'textblob_sentiment', 'roberta_sentiment']
Date available  : True
✅ All data loaded


# CELL 4 — Compute Polarization Score

In [4]:
print("Computing Polarization Scores...")

methods     = [c.replace('_sentiment','') for c in available_cols]
method_names = [m.upper() if m != 'roberta' else 'RoBERTa' for m in methods]

def compute_polarization(df, sent_col, ideology_col='ideology'):
    left  = df[df[ideology_col] == 'Left']
    right = df[df[ideology_col] == 'Right']
    left_pos  = (left[sent_col]  == 'Positive').mean()
    right_pos = (right[sent_col] == 'Positive').mean()
    # Polarization = absolute sentiment gap between Left and Right
    return abs(left_pos - right_pos), left_pos, right_pos

pol_results = {}
for method, name in zip(methods, method_names):
    col = f'{method}_sentiment'
    if col in df.columns:
        pol, lp, rp = compute_polarization(df, col)
        pol_results[name] = {
            'polarization' : round(pol, 4),
            'left_positive': round(lp, 4),
            'right_positive': round(rp, 4),
        }
        print(f"  {name:<12}: Polarization={pol:.4f} | Left+={lp:.3f} | Right+={rp:.3f}")

# Final/majority vote
pol, lp, rp = compute_polarization(df, 'final_sentiment')
pol_results['Majority Vote'] = {
    'polarization' : round(pol, 4),
    'left_positive': round(lp, 4),
    'right_positive': round(rp, 4),
}
print(f"  {'Majority Vote':<12}: Polarization={pol:.4f} | Left+={lp:.3f} | Right+={rp:.3f}")
print("✅ Polarization scores computed")

Computing Polarization Scores...
  VADER       : Polarization=0.0174 | Left+=0.642 | Right+=0.624
  TEXTBLOB    : Polarization=0.0113 | Left+=0.767 | Right+=0.756
  RoBERTa     : Polarization=0.0658 | Left+=0.195 | Right+=0.129
  Majority Vote: Polarization=0.0254 | Left+=0.578 | Right+=0.553
✅ Polarization scores computed


# CELL 5 — Graph 1: Combined Ideology + Sentiment Dashboard

In [5]:
print("Generating Graph 1 — Combined Dashboard...")

fig = plt.figure(figsize=(20, 16), facecolor=BG_COLOR)
fig.suptitle('PoliGraphX — Political Polarization Dashboard',
             fontsize=22, fontweight='bold', color='white', y=0.98)

gs = gridspec.GridSpec(3, 3, figure=fig, hspace=0.45, wspace=0.35)

# ── Panel 1: Ideology Distribution (pie)
ax1 = fig.add_subplot(gs[0, 0])
ideo_counts = df['ideology'].value_counts()
wedges, texts, autotexts = ax1.pie(
    ideo_counts.values,
    labels=ideo_counts.index,
    colors=[LEFT_COLOR, RIGHT_COLOR],
    autopct='%1.1f%%',
    startangle=90,
    wedgeprops={'edgecolor': BG_COLOR, 'linewidth': 3},
    textprops={'color': 'white', 'fontsize': 12}
)
for at in autotexts:
    at.set_fontsize(11)
    at.set_fontweight('bold')
ax1.set_title('Ideology Distribution', fontsize=13, fontweight='bold', color='white', pad=10)

# ── Panel 2: Sentiment by Ideology (grouped bar)
ax2 = fig.add_subplot(gs[0, 1])
ideologies = ['Left', 'Right']
pos_pct = [
    (df[df['ideology']==i]['final_sentiment']=='Positive').mean()*100
    for i in ideologies
]
neg_pct = [100 - p for p in pos_pct]
x = np.arange(2)
w = 0.35
b1 = ax2.bar(x - w/2, pos_pct, w, color=POS_COLOR,  alpha=0.9, label='Positive', edgecolor=BG_COLOR)
b2 = ax2.bar(x + w/2, neg_pct, w, color=NEG_COLOR, alpha=0.9, label='Negative', edgecolor=BG_COLOR)
for bar in list(b1) + list(b2):
    ax2.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
             f'{bar.get_height():.1f}%', ha='center', va='bottom',
             fontsize=10, fontweight='bold', color='white')
ax2.set_xticks(x)
ax2.set_xticklabels(ideologies, fontsize=12)
ax2.set_ylim(0, 120)
ax2.set_ylabel('Percentage (%)', fontsize=11)
ax2.set_title('Sentiment by Ideology\n(Majority Vote)', fontsize=13, fontweight='bold', color='white')
ax2.legend(fontsize=10)
ax2.grid(axis='y', alpha=0.3)

# ── Panel 3: Polarization Score by Method
ax3 = fig.add_subplot(gs[0, 2])
method_labels = list(pol_results.keys())
pol_scores    = [pol_results[m]['polarization'] for m in method_labels]
bar_colors    = [NEUTRAL_COLOR, '#FF6D00', LEFT_COLOR, RIGHT_COLOR][:len(method_labels)]
bars3 = ax3.bar(method_labels, pol_scores, color=bar_colors, alpha=0.9, edgecolor=BG_COLOR)
for bar, val in zip(bars3, pol_scores):
    ax3.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
             f'{val:.4f}', ha='center', va='bottom',
             fontsize=10, fontweight='bold', color='white')
ax3.set_ylabel('Polarization Score', fontsize=11)
ax3.set_title('Polarization Score\nby Method', fontsize=13, fontweight='bold', color='white')
ax3.set_ylim(0, max(pol_scores)*1.3 + 0.01)
ax3.grid(axis='y', alpha=0.3)
ax3.tick_params(axis='x', rotation=15)

# ── Panel 4-6: Sentiment comparison across 3 methods
for idx, (method, name) in enumerate(zip(methods, method_names)):
    ax = fig.add_subplot(gs[1, idx])
    col = f'{method}_sentiment'
    if col not in df.columns:
        continue
    left_pos  = (df[df['ideology']=='Left'][col]=='Positive').mean()*100
    left_neg  = 100 - left_pos
    right_pos = (df[df['ideology']=='Right'][col]=='Positive').mean()*100
    right_neg = 100 - right_pos

    categories = ['Left\nPositive', 'Left\nNegative', 'Right\nPositive', 'Right\nNegative']
    values     = [left_pos, left_neg, right_pos, right_neg]
    colors_bar = [POS_COLOR, NEG_COLOR, POS_COLOR, NEG_COLOR]
    alpha_vals = [0.9, 0.9, 0.6, 0.6]

    bars4 = ax.bar(categories, values, color=colors_bar,
                   alpha=0.85, edgecolor=BG_COLOR, width=0.6)
    for bar, val in zip(bars4, values):
        ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+1,
                f'{val:.1f}%', ha='center', va='bottom',
                fontsize=10, fontweight='bold', color='white')
    ax.set_ylim(0, 120)
    ax.set_ylabel('Percentage (%)', fontsize=10)
    ax.set_title(f'{name} Sentiment\nLeft vs Right', fontsize=12, fontweight='bold', color='white')
    ax.grid(axis='y', alpha=0.3)

# ── Panel 7-9 (bottom row): Tweet volume by ideology + Engagement
ax7 = fig.add_subplot(gs[2, 0])
tweet_counts = df['ideology'].value_counts()
bars7 = ax7.bar(tweet_counts.index, tweet_counts.values,
                color=[LEFT_COLOR, RIGHT_COLOR], alpha=0.9, edgecolor=BG_COLOR, width=0.5)
for bar, val in zip(bars7, tweet_counts.values):
    ax7.text(bar.get_x()+bar.get_width()/2, bar.get_height()+100,
             f'{val:,}', ha='center', va='bottom',
             fontsize=12, fontweight='bold', color='white')
ax7.set_ylabel('Tweet Count', fontsize=11)
ax7.set_title('Tweet Volume\nby Ideology', fontsize=13, fontweight='bold', color='white')
ax7.grid(axis='y', alpha=0.3)

# Engagement comparison
ax8 = fig.add_subplot(gs[2, 1])
eng_cols = [c for c in ['likeCount','retweetCount','replyCount'] if c in df.columns]
if eng_cols:
    eng_data = []
    for col in eng_cols:
        left_avg  = df[df['ideology']=='Left'][col].mean()
        right_avg = df[df['ideology']=='Right'][col].mean()
        eng_data.append([left_avg, right_avg])
    eng_df = pd.DataFrame(eng_data, index=eng_cols, columns=['Left','Right'])
    x8  = np.arange(len(eng_cols))
    w8  = 0.35
    ax8.bar(x8 - w8/2, eng_df['Left'],  w8, color=LEFT_COLOR,  alpha=0.9, label='Left',  edgecolor=BG_COLOR)
    ax8.bar(x8 + w8/2, eng_df['Right'], w8, color=RIGHT_COLOR, alpha=0.9, label='Right', edgecolor=BG_COLOR)
    ax8.set_xticks(x8)
    ax8.set_xticklabels([c.replace('Count','') for c in eng_cols], fontsize=11)
    ax8.set_ylabel('Average Count', fontsize=11)
    ax8.set_title('Avg Engagement\nby Ideology', fontsize=13, fontweight='bold', color='white')
    ax8.legend(fontsize=10)
    ax8.grid(axis='y', alpha=0.3)

# Sentiment + Ideology combined heatmap
ax9 = fig.add_subplot(gs[2, 2])
heatmap_data = pd.crosstab(df['ideology'], df['final_sentiment'])
cmap = LinearSegmentedColormap.from_list('polarity', ['#FF1744','#E040FB','#00B0FF'])
sns.heatmap(heatmap_data, annot=True, fmt=',d', cmap=cmap,
            ax=ax9, linewidths=2, linecolor=BG_COLOR,
            annot_kws={'size': 13, 'weight': 'bold', 'color': 'white'})
ax9.set_title('Ideology × Sentiment\nHeatmap', fontsize=13, fontweight='bold', color='white')
ax9.set_xlabel('Sentiment', fontsize=11)
ax9.set_ylabel('Ideology', fontsize=11)

plt.savefig(os.path.join(OUTPUT_PATH, 'graph11_dashboard.png'),
            dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("✅ Graph 1 — Dashboard saved")


Generating Graph 1 — Combined Dashboard...
✅ Graph 1 — Dashboard saved


# CELL 6 — Graph 2: Polarization Score Over Time

In [6]:
if not has_date:
    print("⚠️ No date column — skipping time graph")
else:
    print("Generating Graph 2 — Polarization Over Time...")

    fig, axes = plt.subplots(1, 3, figsize=(20, 6), facecolor=BG_COLOR)
    fig.suptitle('Political Polarization Score Over Time',
                 fontsize=18, fontweight='bold', color='white')

    all_methods = methods + ['final']
    all_names   = method_names + ['Majority Vote']
    plot_colors = [NEUTRAL_COLOR, '#FF6D00', LEFT_COLOR, RIGHT_COLOR]

    for ax, method, name, color in zip(axes, methods, method_names, plot_colors):
        col = f'{method}_sentiment'
        if col not in df.columns:
            continue

        trend = df.groupby(['month', 'ideology'])[col].apply(
            lambda x: (x == 'Positive').mean() * 100
        ).reset_index()
        trend.columns = ['month', 'ideology', 'positive_pct']
        trend['month_str'] = trend['month'].astype(str)

        left_trend  = trend[trend['ideology'] == 'Left'].sort_values('month_str')
        right_trend = trend[trend['ideology'] == 'Right'].sort_values('month_str')

        ax.plot(left_trend['month_str'],  left_trend['positive_pct'],
                marker='o', color=LEFT_COLOR,  linewidth=2.5, label='Left',  markersize=6)
        ax.plot(right_trend['month_str'], right_trend['positive_pct'],
                marker='s', color=RIGHT_COLOR, linewidth=2.5, label='Right', markersize=6)

        # Shade the gap (polarization area)
        common_months = set(left_trend['month_str']) & set(right_trend['month_str'])
        lt = left_trend[left_trend['month_str'].isin(common_months)].sort_values('month_str')
        rt = right_trend[right_trend['month_str'].isin(common_months)].sort_values('month_str')
        if len(lt) > 0 and len(rt) > 0:
            x_idx = range(len(lt))
            ax.fill_between(lt['month_str'].values,
                            lt['positive_pct'].values,
                            rt['positive_pct'].values,
                            alpha=0.2, color=NEUTRAL_COLOR, label='Gap (Polarization)')

        ax.set_title(f'{name}', fontsize=13, fontweight='bold', color='white')
        ax.set_ylabel('% Positive Tweets', fontsize=11)
        ax.set_xlabel('Month', fontsize=10)
        ax.legend(fontsize=10)
        ax.grid(alpha=0.3)
        ax.tick_params(axis='x', rotation=45)
        ax.set_facecolor(BG_COLOR)

    plt.tight_layout()
    plt.savefig(os.path.join(OUTPUT_PATH, 'graph12_polarization_over_time.png'),
                dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
    plt.show()
    print("✅ Graph 2 — Polarization Over Time saved")


Generating Graph 2 — Polarization Over Time...
✅ Graph 2 — Polarization Over Time saved


# CELL 7 — Graph 3: Key Findings Summary

In [7]:
print("Generating Graph 3 — Key Findings Summary...")

fig, axes = plt.subplots(2, 3, figsize=(20, 12), facecolor=BG_COLOR)
fig.suptitle('PoliGraphX — Key Findings Summary',
             fontsize=20, fontweight='bold', color='white', y=1.01)

# ── Finding 1: Sentiment Agreement across methods
ax = axes[0][0]
agreement_data = []
for method, name in zip(methods, method_names):
    col = f'{method}_sentiment'
    if col in df.columns:
        agree = (df[col] == df['final_sentiment']).mean() * 100
        agreement_data.append((name, agree))

names_a, vals_a = zip(*agreement_data)
bars = ax.barh(names_a, vals_a,
               color=[NEUTRAL_COLOR, '#FF6D00', LEFT_COLOR][:len(names_a)],
               alpha=0.9, edgecolor=BG_COLOR)
for bar, val in zip(bars, vals_a):
    ax.text(bar.get_width() + 0.5, bar.get_y() + bar.get_height()/2,
            f'{val:.1f}%', va='center', fontsize=11, fontweight='bold', color='white')
ax.set_xlim(0, 115)
ax.set_title('Method Agreement\nwith Majority Vote', fontsize=13, fontweight='bold', color='white')
ax.set_xlabel('Agreement (%)', fontsize=11)
ax.grid(axis='x', alpha=0.3)
ax.set_facecolor(BG_COLOR)

# ── Finding 2: Positive sentiment ratio Left vs Right per method
ax = axes[0][1]
left_pos_list  = []
right_pos_list = []
for method, name in zip(methods, method_names):
    col = f'{method}_sentiment'
    if col in df.columns:
        left_pos_list.append((df[df['ideology']=='Left'][col]=='Positive').mean()*100)
        right_pos_list.append((df[df['ideology']=='Right'][col]=='Positive').mean()*100)

x = np.arange(len(method_names))
w = 0.35
ax.bar(x - w/2, left_pos_list,  w, color=LEFT_COLOR,  alpha=0.9, label='Left',  edgecolor=BG_COLOR)
ax.bar(x + w/2, right_pos_list, w, color=RIGHT_COLOR, alpha=0.9, label='Right', edgecolor=BG_COLOR)
ax.set_xticks(x)
ax.set_xticklabels(method_names, fontsize=11)
ax.set_ylabel('% Positive Tweets', fontsize=11)
ax.set_title('Positive Sentiment %\nLeft vs Right by Method', fontsize=13, fontweight='bold', color='white')
ax.legend(fontsize=10)
ax.set_ylim(0, 110)
ax.grid(axis='y', alpha=0.3)
ax.set_facecolor(BG_COLOR)

# ── Finding 3: Polarization scores
ax = axes[0][2]
pol_names  = list(pol_results.keys())
pol_values = [pol_results[m]['polarization'] for m in pol_names]
bar_colors3 = [NEUTRAL_COLOR, '#FF6D00', LEFT_COLOR, RIGHT_COLOR][:len(pol_names)]
bars3 = ax.bar(pol_names, pol_values, color=bar_colors3, alpha=0.9, edgecolor=BG_COLOR)
for bar, val in zip(bars3, pol_values):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.002,
            f'{val:.4f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color='white')
ax.axhline(y=0.1, color='yellow', linestyle='--', alpha=0.7, label='Low polarization threshold')
ax.set_title('Polarization Score\nComparison', fontsize=13, fontweight='bold', color='white')
ax.set_ylabel('Score', fontsize=11)
ax.legend(fontsize=9)
ax.grid(axis='y', alpha=0.3)
ax.tick_params(axis='x', rotation=15)
ax.set_facecolor(BG_COLOR)

# ── Finding 4: Engagement by ideology + sentiment
ax = axes[1][0]
if 'likeCount' in df.columns:
    eng = df.groupby(['ideology', 'final_sentiment'])['likeCount'].mean().unstack()
    eng.plot(kind='bar', ax=ax,
             color=[POS_COLOR, NEG_COLOR],
             alpha=0.9, edgecolor=BG_COLOR, width=0.6)
    ax.set_title('Avg Likes by\nIdeology & Sentiment', fontsize=13, fontweight='bold', color='white')
    ax.set_ylabel('Average Likes', fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Sentiment', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.set_facecolor(BG_COLOR)

# ── Finding 5: Retweet behavior
ax = axes[1][1]
if 'retweetCount' in df.columns:
    rt = df.groupby(['ideology', 'final_sentiment'])['retweetCount'].mean().unstack()
    rt.plot(kind='bar', ax=ax,
            color=[POS_COLOR, NEG_COLOR],
            alpha=0.9, edgecolor=BG_COLOR, width=0.6)
    ax.set_title('Avg Retweets by\nIdeology & Sentiment', fontsize=13, fontweight='bold', color='white')
    ax.set_ylabel('Average Retweets', fontsize=11)
    ax.set_xlabel('')
    ax.tick_params(axis='x', rotation=0)
    ax.legend(title='Sentiment', fontsize=10)
    ax.grid(axis='y', alpha=0.3)
    ax.set_facecolor(BG_COLOR)

# ── Finding 6: Final Polarization Gauge
ax = axes[1][2]
final_pol = pol_results.get('Majority Vote', {}).get('polarization', 0)
left_p    = pol_results.get('Majority Vote', {}).get('left_positive', 0)
right_p   = pol_results.get('Majority Vote', {}).get('right_positive', 0)

# Gauge chart using pie
gauge_sizes  = [final_pol * 100, (1 - final_pol) * 100]
gauge_colors = [RIGHT_COLOR if final_pol > 0.1 else POS_COLOR, '#1E1E3A']
wedges, _    = ax.pie(gauge_sizes, colors=gauge_colors,
                       startangle=90,
                       wedgeprops={'edgecolor': BG_COLOR, 'linewidth': 3})
ax.text(0, 0,    f'{final_pol:.3f}', ha='center', va='center',
        fontsize=22, fontweight='bold', color='white')
ax.text(0, -0.3, 'Polarization\nScore', ha='center', va='center',
        fontsize=11, color='#AAAACC')
ax.text(-0.9, -0.85, f'Left +{left_p*100:.1f}%',
        fontsize=11, color=LEFT_COLOR, fontweight='bold')
ax.text(0.3, -0.85, f'Right +{right_p*100:.1f}%',
        fontsize=11, color=RIGHT_COLOR, fontweight='bold')
ax.set_title('Final Polarization\nGauge', fontsize=13, fontweight='bold', color='white')

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'graph13_key_findings.png'),
            dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("✅ Graph 3 — Key Findings saved")


Generating Graph 3 — Key Findings Summary...
✅ Graph 3 — Key Findings saved


# CELL 8 — Graph 4: Final Conclusions Graph

In [8]:
print("Generating Graph 4 — Final Conclusions...")

fig, axes = plt.subplots(1, 3, figsize=(20, 7), facecolor=BG_COLOR)
fig.suptitle('PoliGraphX — Final Conclusions: Political Polarization Analysis',
             fontsize=17, fontweight='bold', color='white')

# ── Conclusion 1: Overall sentiment breakdown (stacked)
ax = axes[0]
categories = ['Left', 'Right', 'Overall']
pos_vals, neg_vals = [], []
for cat in ['Left', 'Right']:
    sub = df[df['ideology'] == cat]
    pos_vals.append((sub['final_sentiment']=='Positive').mean()*100)
    neg_vals.append((sub['final_sentiment']=='Negative').mean()*100)
pos_vals.append((df['final_sentiment']=='Positive').mean()*100)
neg_vals.append((df['final_sentiment']=='Negative').mean()*100)

x = np.arange(3)
ax.bar(x, pos_vals, color=POS_COLOR, alpha=0.9, label='Positive', edgecolor=BG_COLOR)
ax.bar(x, neg_vals, bottom=pos_vals, color=NEG_COLOR, alpha=0.9, label='Negative', edgecolor=BG_COLOR)
ax.set_xticks(x)
ax.set_xticklabels(categories, fontsize=12)
for i, (p, n) in enumerate(zip(pos_vals, neg_vals)):
    ax.text(i, p/2, f'{p:.1f}%', ha='center', va='center',
            fontsize=12, fontweight='bold', color='black')
    ax.text(i, p + n/2, f'{n:.1f}%', ha='center', va='center',
            fontsize=12, fontweight='bold', color='white')
ax.set_ylabel('Percentage (%)', fontsize=11)
ax.set_title('Final Sentiment\nBreakdown', fontsize=14, fontweight='bold', color='white')
ax.legend(fontsize=11)
ax.set_ylim(0, 115)
ax.grid(axis='y', alpha=0.3)
ax.set_facecolor(BG_COLOR)

# ── Conclusion 2: Radar / Spider of key metrics
ax = axes[1]
metrics_labels = ['Polarization\nScore', 'Left\nPositivity', 'Right\nPositivity',
                  'Method\nAgreement', 'Data\nCoverage']
final_pol   = pol_results.get('Majority Vote', {}).get('polarization', 0)
left_pos_f  = pol_results.get('Majority Vote', {}).get('left_positive', 0)
right_pos_f = pol_results.get('Majority Vote', {}).get('right_positive', 0)
agreement   = np.mean([(df[f'{m}_sentiment']==df['final_sentiment']).mean()
                        for m in methods if f'{m}_sentiment' in df.columns])
coverage    = len(df) / 50000  # normalize to 50K

metric_vals = [final_pol, left_pos_f, right_pos_f, agreement, min(coverage, 1.0)]
bar_colors4 = [RIGHT_COLOR, LEFT_COLOR, RIGHT_COLOR, POS_COLOR, NEUTRAL_COLOR]
bars_c = ax.bar(metrics_labels, metric_vals, color=bar_colors4, alpha=0.9, edgecolor=BG_COLOR)
for bar, val in zip(bars_c, metric_vals):
    ax.text(bar.get_x()+bar.get_width()/2, bar.get_height()+0.01,
            f'{val:.3f}', ha='center', va='bottom',
            fontsize=11, fontweight='bold', color='white')
ax.set_ylim(0, 1.3)
ax.set_ylabel('Score (0-1)', fontsize=11)
ax.set_title('Key Metrics\nSummary', fontsize=14, fontweight='bold', color='white')
ax.grid(axis='y', alpha=0.3)
ax.set_facecolor(BG_COLOR)

# ── Conclusion 3: Method comparison heatmap
ax = axes[2]
heat_data = []
for method, name in zip(methods, method_names):
    col = f'{method}_sentiment'
    if col not in df.columns:
        continue
    row = []
    for ideology in ['Left', 'Right']:
        sub = df[df['ideology'] == ideology]
        row.append((sub[col] == 'Positive').mean() * 100)
    heat_data.append(row)

heat_df = pd.DataFrame(heat_data, index=method_names, columns=['Left', 'Right'])
cmap2   = LinearSegmentedColormap.from_list('lr', [RIGHT_COLOR, '#E040FB', LEFT_COLOR])
sns.heatmap(heat_df, annot=True, fmt='.1f', cmap=cmap2,
            ax=ax, linewidths=2, linecolor=BG_COLOR,
            annot_kws={'size': 14, 'weight': 'bold', 'color': 'white'},
            vmin=0, vmax=100)
ax.set_title('% Positive Tweets\nMethod × Ideology', fontsize=14, fontweight='bold', color='white')
ax.set_xlabel('Ideology', fontsize=12)
ax.set_ylabel('Method', fontsize=12)
ax.set_facecolor(BG_COLOR)

plt.tight_layout()
plt.savefig(os.path.join(OUTPUT_PATH, 'graph14_conclusions.png'),
            dpi=150, bbox_inches='tight', facecolor=BG_COLOR)
plt.show()
print("✅ Graph 4 — Final Conclusions saved")


Generating Graph 4 — Final Conclusions...
✅ Graph 4 — Final Conclusions saved


# CELL 9 — Generate Final Summary Report (CSV)

In [9]:
print("Generating Final Summary Report...")

report_rows = []

# Dataset stats
report_rows += [
    ('DATASET', 'Total Tweets',        f"{len(df):,}"),
    ('DATASET', 'Left Tweets',         f"{(df['ideology']=='Left').sum():,}"),
    ('DATASET', 'Right Tweets',        f"{(df['ideology']=='Right').sum():,}"),
    ('DATASET', 'Unique Users',        f"{df['username'].nunique():,}" if 'username' in df.columns else 'N/A'),
]

# Sentiment stats
for method, name in zip(methods + ['final'], method_names + ['Majority Vote']):
    col = f'{method}_sentiment' if method != 'final' else 'final_sentiment'
    if col not in df.columns:
        continue
    pos = (df[col]=='Positive').mean()*100
    neg = 100 - pos
    report_rows += [
        (f'SENTIMENT ({name})', 'Overall Positive', f"{pos:.2f}%"),
        (f'SENTIMENT ({name})', 'Overall Negative', f"{neg:.2f}%"),
        (f'SENTIMENT ({name})', 'Left Positive',    f"{(df[df['ideology']=='Left'][col]=='Positive').mean()*100:.2f}%"),
        (f'SENTIMENT ({name})', 'Right Positive',   f"{(df[df['ideology']=='Right'][col]=='Positive').mean()*100:.2f}%"),
    ]

# Polarization
for method_name, vals in pol_results.items():
    report_rows.append(('POLARIZATION', f'{method_name} Score', f"{vals['polarization']:.4f}"))

# Network stats (if available)
if net_summary is not None:
    for _, row in net_summary.iterrows():
        report_rows.append(('NETWORK', row['Metric'], str(row['Value'])))

report_df   = pd.DataFrame(report_rows, columns=['Category', 'Metric', 'Value'])
report_path = os.path.join(OUTPUT_PATH, 'final_report.csv')
report_df.to_csv(report_path, index=False)

print(report_df.to_string(index=False))
print(f"\n✅ Report saved → {report_path}")
"""


# ============================================================
# CELL 10 — Final Summary
# ============================================================
"""
print("=" * 60)
print("  POLIGRAPHX — COMPLETE ✅")
print("=" * 60)
print()
print("All outputs saved to:", OUTPUT_PATH)
print()
print("  NOTEBOOK 2 OUTPUTS:")
print("  ✅ graph1_model_comparison.png")
print("  ✅ graph2_confusion_matrix.png")
print("  ✅ classification_report.csv")
print()
print("  NOTEBOOK 3 OUTPUTS:")
print("  ✅ graph3_sentiment_distribution.png")
print("  ✅ graph4_sentiment_over_time.png")
print("  ✅ graph5_wordclouds.png")
print("  ✅ graph6_confidence_scores.png")
print()
print("  NOTEBOOK 4 OUTPUTS:")
print("  ✅ graph7_interaction_network.png")
print("  ✅ graph8_echo_chamber.png")
print("  ✅ graph9_top_users.png")
print("  ✅ graph10_communities.png")
print()
print("  NOTEBOOK 5 OUTPUTS (FINAL):")
print("  ✅ graph11_dashboard.png")
print("  ✅ graph12_polarization_over_time.png")
print("  ✅ graph13_key_findings.png")
print("  ✅ graph14_conclusions.png")
print("  ✅ final_report.csv")
print()
print("=" * 60)
final_pol = pol_results.get('Majority Vote', {}).get('polarization', 0)
print(f"  FINAL POLARIZATION SCORE : {final_pol:.4f}")
if final_pol > 0.15:
    print("  CONCLUSION: Strong political polarization detected!")
elif final_pol > 0.08:
    print("  CONCLUSION: Moderate political polarization detected.")
else:
    print("  CONCLUSION: Low political polarization detected.")
print("=" * 60)


Generating Final Summary Report...
                 Category                      Metric    Value
                  DATASET                Total Tweets   42,110
                  DATASET                 Left Tweets   15,244
                  DATASET                Right Tweets   26,866
                  DATASET                Unique Users   28,200
        SENTIMENT (VADER)            Overall Positive   63.07%
        SENTIMENT (VADER)            Overall Negative   36.93%
        SENTIMENT (VADER)               Left Positive   64.18%
        SENTIMENT (VADER)              Right Positive   62.44%
     SENTIMENT (TEXTBLOB)            Overall Positive   76.00%
     SENTIMENT (TEXTBLOB)            Overall Negative   24.00%
     SENTIMENT (TEXTBLOB)               Left Positive   76.72%
     SENTIMENT (TEXTBLOB)              Right Positive   75.59%
      SENTIMENT (RoBERTa)            Overall Positive   15.32%
      SENTIMENT (RoBERTa)            Overall Negative   84.68%
      SENTIMENT (RoB